[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_POS.ipynb)

# Benchmark: POS Tagging

Scores a pretrained POS tagger's accuracy against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="pos")`.

**Dataset**: [Universal Dependencies English EWT](https://github.com/UniversalDependencies/UD_English-EWT),
test split (CC BY-SA 4.0), `.conllu` format.

**Model**: `PerceptronModel.pretrained()`, Spark NLP's default pretrained English POS tagger.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.training import CoNLL
from sparknlp.annotator import PerceptronModel
from pyspark.ml import Pipeline
from pyspark.sql.functions import expr
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

We use the `XPOS` column (Penn Treebank-style tags like `NNP`/`VBD`), since that's the tag set
Spark NLP's pretrained English `PerceptronModel` was trained on -- the `UPOS` (universal) column
uses a different, incompatible tag set that would make any model look wrong regardless of its
real accuracy (see the same lesson in the NER notebook).

In [8]:
import urllib.request

url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu"
conllu_text = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

sentences = []
current = []
for line in conllu_text.split("\n"):
    if line.startswith("#"):
        continue
    if line.strip() == "":
        if current:
            sentences.append(current)
            current = []
        continue
    cols = line.split("\t")
    if "-" in cols[0] or "." in cols[0]:
        continue  # multiword-token / empty-node lines, not real tokens
    current.append((cols[1], cols[4]))  # (FORM, XPOS)
if current:
    sentences.append(current)

print(len(sentences), "sentences")
print(sentences[0])

2077 sentences
[('What', 'WP'), ('if', 'IN'), ('Google', 'NNP'), ('Morphed', 'VBD'), ('Into', 'IN'), ('GoogleOS', 'NNP'), ('?', '.')]

Spark NLP's `CoNLL()` reader expects the classic CoNLL-2003 4-column layout, not `.conllu`
-- so we convert once, reusing this proven reader instead of hand-building Spark NLP's
annotation structs. This also guarantees the `token` column it produces stays perfectly aligned
with our gold tags, since both come from the exact same tokenization.

In [10]:
conll_path = "/tmp/ud_english_ewt_test.conll2003"
with open(conll_path, "w") as f:
    for sent in sentences:
        for form, xpos in sent:
            f.write(f"{form} {xpos} O O\n")
        f.write("\n")

gold_data = CoNLL().readDataset(spark, conll_path)
gold_data = gold_data.withColumn("gold_tags", expr("transform(pos, x -> x.result)"))
gold_data.select("text", "gold_tags").show(3, truncate=80)
print(gold_data.count(), "sentences loaded")

+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            text|                                                                       gold_tags|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                          What if Google Morphed Into GoogleOS ?|                                                  [WP, IN, NNP, VBD, IN, NNP, .]|
|What if Google expanded on its search - engine ( and now e-mail ) wares into ...|[WP, IN, NNP, VBD, IN, PRP$, NN, HYPH, NN, -LRB-, CC, RB, NN, -RRB-, NNS, IN,...|
|                                      [ via Microsoft Watch from Mary Jo Foley ]|                                 [-LRB-, IN, NNP, NNP, IN, NNP, NNP, NNP, -RRB-]|
+---------------

## 2. Build the pipeline

In [12]:
pos_model = PerceptronModel.pretrained() \
    .setInputCols(["sentence", "token"]).setOutputCol("pos_pred")

pipeline = Pipeline(stages=[pos_model])
pipeline_model = pipeline.fit(gold_data)

pos_anc download started this may take some time.
Approximate size to download 3.9 MB

[ | ]
[ / ]
[ — ]
[ \ ]
[ | ]
[OK!]

## 3. Run the benchmark

In [14]:
report = Benchmark.evaluate(pipeline_model, gold_data, task="pos", label_col="gold_tags")
print(report)

pos accuracy (n=25094): accuracy=0.8537, weightedF1=0.8490, weightedPrecision=0.8563, weightedRecall=0.8537
  #: f1=0.0000, precision=0.0000, recall=0.0000
  $: f1=0.9836, precision=0.9677, recall=1.0000
  '': f1=0.6559, precision=0.5094, recall=0.9205
  (: f1=0.0000, precision=0.0000, recall=0.0000
  ): f1=0.0000, precision=0.0000, recall=0.0000
  ,: f1=0.9176, precision=1.0000, recall=0.8478
  -: f1=0.0000, precision=0.0000, recall=0.0000
  -LRB-: f1=0.0000, precision=0.0000, recall=0.0000
  -RRB-: f1=0.0000, precision=0.0000, recall=0.0000
  .: f1=0.9890, precision=0.9910, recall=0.9869
  ...: f1=0.0000, precision=0.0000, recall=0.0000
  :: f1=0.5389, precision=0.3731, recall=0.9700
  ADD: f1=0.0000, precision=0.0000, recall=0.0000
  AFX: f1=0.0000, precision=0.0000, recall=0.0000
  CC: f1=0.9925, precision=0.9973, recall=0.9878
  CD: f1=0.8480, precision=0.8795, recall=0.8187
  DT: f1=0.9834, precision=0.9797, recall=0.9872
  EX: f1=0.8846, precision=0.8214, recall=0.9583
  FW: f1=

## Reading the result

Per-item tag accuracy plus a per-tag precision/recall/F1 breakdown -- useful for spotting which
specific tags a model confuses (e.g. proper nouns vs. common nouns), not just an aggregate
number.